# Notebook 21 — Project Benchmark, Routing, and Structured Outputs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/21_project_benchmark_routing_and_structured_outputs.ipynb)

The second capstone notebook turns the Castalia Runtime prototype into a measurable routing system. We compare candidate runtimes, evaluate structured-output reliability, and turn benchmark results into explicit path-selection policy for Castalia Mentor.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Capstone continuation**
- Understand **Step 1: Prepare helpers and artifact paths**
- Understand **Step 2: Define benchmark workloads and candidate runtimes**
- Understand **Step 3: Simulate benchmark runs**


## What you will build

- a benchmark table across local and optimized runtimes
- a structured-output contract for Castalia Mentor responses
- routing logic that uses latency, throughput, and contract pass rates
- policy gates for safety, structured-output reliability, and SLA compliance
- an exportable routing manifest grounded in benchmark evidence

## Capstone continuation

Notebook 20 defined the serving contract and the first runtime skeleton. This notebook adds the selection logic that lets Castalia Mentor choose a route with evidence instead of intuition.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Prepare helpers and artifact paths

We keep the benchmark notebook lightweight: deterministic simulation, small helper functions, and explicit artifact exports.

In [ ]:
random.seed(21)

ARTIFACT_DIR = ARTIFACT_ROOT / "project_21_benchmark_routing"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def percentile(values, pct):
    ordered = sorted(float(value) for value in values)
    if not ordered:
        return 0.0
    rank = (len(ordered) - 1) * (pct / 100)
    lower = math.floor(rank)
    upper = math.ceil(rank)
    if lower == upper:
        return ordered[int(rank)]
    weight = rank - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define benchmark workloads and candidate runtimes

The routing layer only makes sense if the workload mix is concrete. We benchmark several Castalia Mentor request families against a local notebook path and two optimized open-source runtimes.

In [ ]:
benchmark_workloads = [
    {"workload": "mentor_chat_short", "prompt_tokens": 220, "output_tokens": 120, "require_json": False, "safety_level": "standard"},
    {"workload": "mentor_extract_json", "prompt_tokens": 420, "output_tokens": 90, "require_json": True, "safety_level": "strict"},
    {"workload": "mentor_plan_rollout", "prompt_tokens": 560, "output_tokens": 180, "require_json": True, "safety_level": "strict"},
    {"workload": "mentor_policy_long", "prompt_tokens": 980, "output_tokens": 220, "require_json": False, "safety_level": "strict"},
]

candidate_runtimes = {
    "local_notebook": {
        "backend": "llama.cpp",
        "queue_ms": 42,
        "prefill_per_token_ms": 0.88,
        "prefill_bias_ms": 120,
        "decode_per_token_ms": 2.35,
        "decode_bias_ms": 160,
        "throughput_tps": 28,
        "quality_mean": 0.86,
        "structured_pass_rate": 0.76,
        "policy_pass_rate": 0.975,
        "cost_per_1k_output": 0.9,
    },
    "optimized_server": {
        "backend": "vLLM",
        "queue_ms": 18,
        "prefill_per_token_ms": 0.32,
        "prefill_bias_ms": 92,
        "decode_per_token_ms": 1.05,
        "decode_bias_ms": 110,
        "throughput_tps": 86,
        "quality_mean": 0.89,
        "structured_pass_rate": 0.94,
        "policy_pass_rate": 0.988,
        "cost_per_1k_output": 2.4,
    },
    "advanced_router": {
        "backend": "SGLang",
        "queue_ms": 22,
        "prefill_per_token_ms": 0.28,
        "prefill_bias_ms": 88,
        "decode_per_token_ms": 0.94,
        "decode_bias_ms": 102,
        "throughput_tps": 94,
        "quality_mean": 0.895,
        "structured_pass_rate": 0.965,
        "policy_pass_rate": 0.991,
        "cost_per_1k_output": 2.8,
    },
}

runtime_profile_df = pd.DataFrame([
    {"runtime": runtime_name, **profile}
    for runtime_name, profile in candidate_runtimes.items()
])
runtime_profile_df

## Step 3: Simulate benchmark runs

The benchmark harness below models queue time, prefill, decode, policy compliance, and structured-output reliability. The point is not perfect realism; the point is to preserve the relationships that good routing logic depends on.

In [ ]:
def run_benchmark(workload, runtime_name, repeats=40):
    profile = candidate_runtimes[runtime_name]
    rows = []
    for index in range(repeats):
        rng = random.Random(f"{workload['workload']}|{runtime_name}|{index}")
        queue_ms = max(4.0, rng.gauss(profile["queue_ms"], profile["queue_ms"] * 0.20))
        prefill_ms = max(20.0, workload["prompt_tokens"] * profile["prefill_per_token_ms"] + rng.gauss(profile["prefill_bias_ms"], 12))
        decode_ms = max(40.0, workload["output_tokens"] * profile["decode_per_token_ms"] + rng.gauss(profile["decode_bias_ms"], 18))
        total_ms = round(queue_ms + prefill_ms + decode_ms, 2)
        schema_pass = rng.random() < (profile["structured_pass_rate"] if workload["require_json"] else 0.995)
        policy_pass = rng.random() < profile["policy_pass_rate"]
        quality_score = round(max(0.0, min(1.0, rng.gauss(profile["quality_mean"], 0.02))), 3)
        rows.append({
            "workload": workload["workload"],
            "runtime": runtime_name,
            "backend": profile["backend"],
            "require_json": workload["require_json"],
            "safety_level": workload["safety_level"],
            "total_ms": total_ms,
            "schema_pass": schema_pass,
            "policy_pass": policy_pass,
            "quality_score": quality_score,
            "throughput_tps": profile["throughput_tps"],
            "cost_per_1k_output": profile["cost_per_1k_output"],
        })
    return pd.DataFrame(rows)

benchmark_df = pd.concat([
    run_benchmark(workload, runtime_name)
    for workload in benchmark_workloads
    for runtime_name in candidate_runtimes
], ignore_index=True)

benchmark_df.head(12)

## Step 4: Summarize the runtime comparison

Routing should read compact benchmark summaries, not raw request logs. We aggregate latency, contract success, quality, and cost into one table per workload and runtime.

In [ ]:
summary_rows = []
for (runtime_name, workload_name), frame in benchmark_df.groupby(["runtime", "workload"]):
    summary_rows.append({
        "runtime": runtime_name,
        "workload": workload_name,
        "p50_latency_ms": round(percentile(frame["total_ms"], 50), 2),
        "p95_latency_ms": round(percentile(frame["total_ms"], 95), 2),
        "schema_pass_rate": round(float(frame["schema_pass"].mean()), 3),
        "policy_pass_rate": round(float(frame["policy_pass"].mean()), 3),
        "quality_score": round(float(frame["quality_score"].mean()), 3),
        "throughput_tps": round(float(frame["throughput_tps"].mean()), 1),
        "cost_per_1k_output": round(float(frame["cost_per_1k_output"].mean()), 2),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["workload", "p50_latency_ms"])
summary_df

## Step 5: Define the structured-output contract

Benchmark speed alone is not enough for Castalia Mentor. Some routes must emit stable JSON so downstream tooling can trust the answer shape.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Citation(BaseModel):
    title: str
    confidence: float

class MentorStructuredAnswer(BaseModel):
    request_id: str
    answer: str
    route: str
    citations: list[Citation] = Field(default_factory=list)
    escalation: bool = False

structured_contract_schema = MentorStructuredAnswer.model_json_schema()
print(json.dumps(structured_contract_schema, indent=2)[:1600])

## Step 6: Measure structured-output pass rates

This step turns structured-output correctness into a routing input. A fast runtime that cannot keep the schema intact should lose traffic for schema-sensitive requests.

In [ ]:
def emit_candidate_payload(workload, runtime_name, index):
    profile = candidate_runtimes[runtime_name]
    rng = random.Random(f"structured|{workload['workload']}|{runtime_name}|{index}")
    if workload["require_json"] and rng.random() > profile["structured_pass_rate"]:
        return {"request_id": f"{runtime_name}-{index}", "answer": 17, "route": runtime_name}
    citation = {"title": "systems-runtime-notes", "confidence": round(max(0.5, min(0.99, rng.gauss(0.84, 0.06))), 2)}
    return {
        "request_id": f"{runtime_name}-{index}",
        "answer": "Structured Castalia Mentor answer",
        "route": runtime_name,
        "citations": [citation],
        "escalation": workload["safety_level"] == "strict" and rng.random() < 0.15,
    }

structured_rows = []
for workload in benchmark_workloads:
    if not workload["require_json"]:
        continue
    for runtime_name in candidate_runtimes:
        accepted = 0
        for index in range(60):
            payload = emit_candidate_payload(workload, runtime_name, index)
            try:
                MentorStructuredAnswer.model_validate(payload)
                accepted += 1
            except ValidationError:
                pass
        structured_rows.append({
            "workload": workload["workload"],
            "runtime": runtime_name,
            "contract_pass_rate": round(accepted / 60, 3),
        })

structured_summary_df = pd.DataFrame(structured_rows).sort_values(["workload", "contract_pass_rate"], ascending=[True, False])
structured_summary_df

## Step 7: Build benchmark-driven routing logic

A routing function should combine performance evidence with contract requirements. Here we use latency, quality, policy success, throughput, and structured-output pass rates.

In [ ]:
summary_lookup = {(row["runtime"], row["workload"]): row for row in summary_df.to_dict(orient="records")}
structured_lookup = {(row["workload"], row["runtime"]): row["contract_pass_rate"] for row in structured_summary_df.to_dict(orient="records")}

def choose_runtime(workload_name, require_json=False, strict_policy=False, concurrency=8):
    candidates = []
    for runtime_name in candidate_runtimes:
        row = dict(summary_lookup[(runtime_name, workload_name)])
        row["contract_pass_rate"] = structured_lookup.get((workload_name, runtime_name), 1.0)
        candidates.append(row)
    candidate_df = pd.DataFrame(candidates)
    filtered_df = candidate_df.copy()
    if require_json:
        filtered_df = filtered_df[filtered_df["contract_pass_rate"] >= 0.90]
    if strict_policy:
        filtered_df = filtered_df[filtered_df["policy_pass_rate"] >= 0.97]
    if filtered_df.empty:
        filtered_df = candidate_df.copy()
    if concurrency >= 24:
        selected = filtered_df.sort_values(["throughput_tps", "p95_latency_ms"], ascending=[False, True]).iloc[0]
    else:
        selected = filtered_df.sort_values(["p50_latency_ms", "cost_per_1k_output"], ascending=[True, True]).iloc[0]
    return selected.to_dict()

choose_runtime("mentor_extract_json", require_json=True, strict_policy=True, concurrency=16)

## Step 8: Add policy gates

Selection is not the same as permission. Policy gates ensure a route only receives traffic if it stays inside the latency budget and satisfies quality and contract minimums.

In [ ]:
def policy_gate(selection, latency_budget_ms, minimum_quality=0.84, minimum_contract_pass=0.90):
    reasons = []
    if selection["p95_latency_ms"] > latency_budget_ms:
        reasons.append("latency")
    if selection["quality_score"] < minimum_quality:
        reasons.append("quality")
    if selection["contract_pass_rate"] < minimum_contract_pass:
        reasons.append("contract")
    if selection["policy_pass_rate"] < 0.97:
        reasons.append("policy")
    if not reasons:
        status = "allow"
    elif reasons == ["latency"]:
        status = "review"
    else:
        status = "block"
    return {
        "runtime": selection["runtime"],
        "status": status,
        "reasons": reasons,
        "p95_latency_ms": selection["p95_latency_ms"],
        "quality_score": selection["quality_score"],
        "contract_pass_rate": selection["contract_pass_rate"],
    }

gate_examples = [
    policy_gate(choose_runtime("mentor_chat_short", require_json=False, strict_policy=False, concurrency=6), latency_budget_ms=950),
    policy_gate(choose_runtime("mentor_extract_json", require_json=True, strict_policy=True, concurrency=16), latency_budget_ms=980),
    policy_gate(choose_runtime("mentor_policy_long", require_json=False, strict_policy=True, concurrency=28), latency_budget_ms=1500),
]

gate_df = pd.DataFrame(gate_examples)
gate_df

## Step 9: Route realistic benchmark-driven scenarios

Now we can route a few Castalia Mentor scenarios and inspect which runtime wins once workload shape, concurrency, and output constraints are all considered together.

In [ ]:
routing_scenarios = [
    {"scenario": "default mentor chat", "workload": "mentor_chat_short", "require_json": False, "strict_policy": False, "concurrency": 6, "latency_budget_ms": 950},
    {"scenario": "structured extraction", "workload": "mentor_extract_json", "require_json": True, "strict_policy": True, "concurrency": 16, "latency_budget_ms": 980},
    {"scenario": "planning at moderate load", "workload": "mentor_plan_rollout", "require_json": True, "strict_policy": True, "concurrency": 18, "latency_budget_ms": 1200},
    {"scenario": "long policy summary under load", "workload": "mentor_policy_long", "require_json": False, "strict_policy": True, "concurrency": 28, "latency_budget_ms": 1500},
]

scenario_rows = []
for scenario in routing_scenarios:
    selection = choose_runtime(
        scenario["workload"],
        require_json=scenario["require_json"],
        strict_policy=scenario["strict_policy"],
        concurrency=scenario["concurrency"],
    )
    gate = policy_gate(selection, latency_budget_ms=scenario["latency_budget_ms"])
    scenario_rows.append({
        "scenario": scenario["scenario"],
        "runtime": selection["runtime"],
        "p50_latency_ms": selection["p50_latency_ms"],
        "contract_pass_rate": selection["contract_pass_rate"],
        "policy_status": gate["status"],
    })

scenario_df = pd.DataFrame(scenario_rows)
scenario_df

## Step 10: Export the routing manifest and benchmark evidence

The point of the routing notebook is to leave behind a manifest the deployment handoff can trust. We therefore export both the summarized benchmark tables and the routing policy assumptions.

In [ ]:
routing_manifest = {
    "project": "Castalia Runtime",
    "service": "Castalia Mentor",
    "selection_policy": {
        "require_json_contract_pass_min": 0.90,
        "policy_pass_rate_min": 0.97,
        "concurrency_switch_threshold": 24,
    },
    "candidate_runtimes": candidate_runtimes,
    "benchmark_summary": summary_df.to_dict(orient="records"),
    "structured_summary": structured_summary_df.to_dict(orient="records"),
    "scenario_routes": scenario_df.to_dict(orient="records"),
}

summary_df.to_csv(ARTIFACT_DIR / "benchmark_summary.csv", index=False)
structured_summary_df.to_csv(ARTIFACT_DIR / "structured_summary.csv", index=False)
gate_df.to_csv(ARTIFACT_DIR / "policy_gate_examples.csv", index=False)
scenario_df.to_csv(ARTIFACT_DIR / "scenario_routes.csv", index=False)
write_json(ARTIFACT_DIR / "routing_manifest.json", routing_manifest)
print("Wrote benchmark artifacts:", sorted(path.name for path in ARTIFACT_DIR.iterdir()))

## Step 11: Score why one route wins

A compact composite score can help you explain routing behavior to operators. It is not a substitute for the raw metrics, but it makes the tradeoff visible in one number.

In [ ]:
score_rows = []
for workload in benchmark_workloads:
    for runtime_name in candidate_runtimes:
        summary_row = dict(summary_lookup[(runtime_name, workload["workload"])])
        contract_pass_rate = structured_lookup.get((workload["workload"], runtime_name), 1.0)
        composite_score = round(
            (1000 / summary_row["p50_latency_ms"])
            + summary_row["quality_score"] * 3.0
            + contract_pass_rate * 2.5
            + summary_row["policy_pass_rate"] * 2.0
            + (summary_row["throughput_tps"] / 60)
            - (summary_row["cost_per_1k_output"] / 6),
            3,
        )
        score_rows.append({
            "workload": workload["workload"],
            "runtime": runtime_name,
            "contract_pass_rate": contract_pass_rate,
            "composite_score": composite_score,
        })

score_df = pd.DataFrame(score_rows).sort_values(["workload", "composite_score"], ascending=[True, False])
score_df

## Recap

You now have benchmark evidence, routing logic, structured-output checks, and policy gates for Castalia Runtime. Notebook 22 turns these outputs into a deployment handoff with manifests, regression gates, and operator-ready release steps.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** benchmark with a different model size and compare throughput. Document what changes and why.

**Exercise 2 — Build:** add a monitoring metric to the serving pipeline. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** simulate a failure scenario and verify the system recovers.

## 📝 Key Takeaways

- **What you will build** — revisit this section for reference
- **Capstone continuation** — revisit this section for reference
- **Step 1: Prepare helpers and artifact paths** — revisit this section for reference
- **Step 2: Define benchmark workloads and candidate runtimes** — revisit this section for reference
- **Step 3: Simulate benchmark runs** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the systems/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [20 Project Runtime Prototype](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/20_project_runtime_prototype.ipynb) | ➡️ **Next:** [22 Project Deployment Handoff](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/22_project_deployment_handoff.ipynb)